In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df_text = pd.read_csv('drive/MyDrive/Colab Notebooks/NLP/processed_v2.xls')
df_text['statement_1'] = df_text['statement_1'].str.lower()
df_text['statement_2'] = df_text['statement_2'].str.lower()
df_labels = pd.read_csv('drive/MyDrive/Colab Notebooks/NLP/labels.xls')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load_dict(path):
    with open(path, 'r', encoding='utf-8') as f:
        return set([line.strip().lower() for line in f if line.strip()])

countries = load_dict('drive/MyDrive/Colab Notebooks/NLP/resources/countries.txt')
continents = load_dict('drive/MyDrive/Colab Notebooks/NLP/resources/continents.txt')
persons = load_dict('drive/MyDrive/Colab Notebooks/NLP/resources/persons.txt')

In [ ]:
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text

In [ ]:
def extract_locations(text):
    text_norm = normalize_text(text)
    results = []

    for word in countries.union(continents):
        start = text_norm.find(word)
        if start != -1:
            results.append({
                'field_type': 'LOCATION',
                'value': word,
                'start_char': start,
                'end_char': start + len(word),
                'method': 'dict_geo_v1'
            })

    return results

In [ ]:
def extract_persons(text):
    text_norm = normalize_text(text)
    results = []

    for name in persons:
        start = text_norm.find(name)
        if start != -1:
            results.append({
                'field_type': 'PERSON',
                'value': name,
                'start_char': start+1,
                'end_char': start + len(name)+1,
                'method': 'dict_person_v1'
            })

    return results

In [ ]:
def extract_all(text):
    return (
        extract_locations(text) +
        extract_persons(text)
    )

In [ ]:
sample = df_text.iloc[8]['statement_1']

print(sample)
print(extract_all(sample))

том має роботу.
[{'field_type': 'PERSON', 'value': 'том', 'start_char': 1, 'end_char': 4, 'method': 'dict_person_v1'}]


In [ ]:
df_sample = df_text.sample(50, random_state=42).reset_index(drop=True)
df_sample['text_id'] = df_sample.index
df_sample['text'] = df_sample['statement_1'] + ' ' + df_sample['statement_2']
df_sample['text'] = df_sample['text'].str.lower()

df_sample = df_sample[['text_id', 'text']]
df_sample.head()


df_sample.to_csv('data_sample_for_labeling.csv', index=False)

In [ ]:
gold_df = pd.read_csv('drive/MyDrive/Colab Notebooks/NLP/resources/labeled_gold.csv')

gold_df.to_json(
    'drive/MyDrive/Colab Notebooks/NLP/resources/gold_ie.jsonl',
    orient='records',
    lines=True,
    force_ascii=False
)

In [ ]:
gold = pd.read_json('drive/MyDrive/Colab Notebooks/NLP/resources/gold_ie.jsonl', lines=True)
gold['span_text'] = gold['span_text'].str.lower()
pred = []

for _, row in gold[['text_id', 'text']].drop_duplicates().iterrows():
    extracted = extract_all(row['text'])

    for e in extracted:
        pred.append({
            'text_id': row['text_id'],
            'field_type': e['field_type'],
            'value': e['value'],
            'start_char': e['start_char'],
            'end_char': e['end_char']
        })

pred = pd.DataFrame(pred)


gold_match = gold[['text_id','field_type','start_char','end_char']].dropna()

pred['is_correct'] = pred.merge(
    gold_match,
    on=['text_id','field_type','start_char','end_char'],
    how='left',
    indicator=True
)['_merge'] == 'both'


precision_table = pred.groupby('field_type').agg(
    total_extractions=('is_correct', 'count'),
    correct_extractions=('is_correct', 'sum')
)

precision_table['precision'] = (
    precision_table['correct_extractions'] /
    precision_table['total_extractions']
)

precision_table

,total_extractions,correct_extractions,precision
field_type,,,
PERSON,22,14,0.636364


In [ ]:
gold.head()

,text_id,text,field_type,span_text,start_char,end_char,normalized_value
0,0,У Тома є мотоцикл. Я вивчаю фарсі.,PERSON,тома,3.0,7.0,тома
1,1,Вона має білого кота. Я не люблю каву.,None,None,NaN,NaN,None
2,2,Схоже на апельсин. Маю ще одне запитання.,None,None,NaN,NaN,None
3,3,Вона має білого кота. Він має собаку.,None,None,NaN,NaN,None
4,4,У мене є бібліотечна картка. Вчора ввечері мен...,None,None,NaN,NaN,None


In [ ]:
overall_precision = pred['is_correct'].sum() / len(pred)
print('overall precision:', overall_precision)

overall precision: 0.6363636363636364


In [ ]:
fp = pred[pred['is_correct'] == False].copy()

fp = fp.merge(
    gold[['text_id','text']].drop_duplicates(),
    on='text_id',
    how='left'
)

fp.head(10)

,text_id,field_type,value,start_char,end_char,is_correct,text
0,0,PERSON,том,3,6,False,У Тома є мотоцикл. Я вивчаю фарсі.
1,10,PERSON,том,36,39,False,Оранжевий - мій улюблений колір. У Тома є час.
2,15,PERSON,том,44,47,False,Я вивчаю ісландську мову. Мері подобається Тому.
3,18,PERSON,том,3,6,False,У Тома брат та сестра. Том має брата і сестру.
4,20,PERSON,том,23,26,False,"Можливо. Цікаво, чи у Тома є права."
5,28,PERSON,том,11,14,False,Скільки у Тома гітар? Скільки Том має гітар?
6,38,PERSON,том,3,6,False,У Тома аутизм та синдром порушення активності ...
7,39,PERSON,том,3,6,False,У Тома багато справ. Він не дурний.


In [ ]:
fp = pred[pred['is_correct'] == False].copy()

fp = fp.merge(
    gold[['text_id','text']].drop_duplicates(),
    on='text_id',
    how='left'
)

fp.head(10)

,text_id,field_type,value,start_char,end_char,is_correct,text
0,0,PERSON,том,3,6,False,У Тома є мотоцикл. Я вивчаю фарсі.
1,10,PERSON,том,36,39,False,Оранжевий - мій улюблений колір. У Тома є час.
2,15,PERSON,том,44,47,False,Я вивчаю ісландську мову. Мері подобається Тому.
3,18,PERSON,том,3,6,False,У Тома брат та сестра. Том має брата і сестру.
4,20,PERSON,том,23,26,False,"Можливо. Цікаво, чи у Тома є права."
5,28,PERSON,том,11,14,False,Скільки у Тома гітар? Скільки Том має гітар?
6,38,PERSON,том,3,6,False,У Тома аутизм та синдром порушення активності ...
7,39,PERSON,том,3,6,False,У Тома багато справ. Він не дурний.


In [ ]:
import re

def extract_persons(text):
    results = []
    text_norm = normalize_text(text)

    for name in persons:
        # allow simple Ukrainian endings
        pattern = r'\b' + re.escape(name) + r'(а|у|ом|і|е|ою)?\b'

        for match in re.finditer(pattern, text_norm):
            results.append({
                'field_type': 'PERSON',
                'value': name,  # normalized
                'start_char': match.start()+1,
                'end_char': match.end()+1,
                'method': 'dict_person_v3'
            })

    return results

In [ ]:
gold = pd.read_json('drive/MyDrive/Colab Notebooks/NLP/resources/gold_ie.jsonl', lines=True)

pred = []

for _, row in gold[['text_id', 'text']].drop_duplicates().iterrows():
    extracted = extract_all(row['text'])

    for e in extracted:
        pred.append({
            'text_id': row['text_id'],
            'field_type': e['field_type'],
            'value': e['value'],
            'start_char': e['start_char'],
            'end_char': e['end_char']
        })

pred = pd.DataFrame(pred)


gold_match = gold[['text_id','field_type','start_char','end_char']].dropna()

pred['is_correct'] = pred.merge(
    gold_match,
    on=['text_id','field_type','start_char','end_char'],
    how='left',
    indicator=True
)['_merge'] == 'both'


precision_table = pred.groupby('field_type').agg(
    total_extractions=('is_correct', 'count'),
    correct_extractions=('is_correct', 'sum')
)

precision_table['precision'] = (
    precision_table['correct_extractions'] /
    precision_table['total_extractions']
)

precision_table

,total_extractions,correct_extractions,precision
field_type,,,
PERSON,31,21,0.677419


In [ ]:
overall_precision = pred['is_correct'].mean()
print(overall_precision)
fp = pred[pred['is_correct'] == False].copy()

fp = fp.merge(
    gold[['text_id','text']].drop_duplicates(),
    on='text_id',
    how='left'
)

fp.head(10)

0.6774193548387096


,text_id,field_type,value,start_char,end_char,is_correct,text
0,11,PERSON,том,16,19,False,Том плакатиме. Том буде плакати.
1,15,PERSON,том,44,48,False,Я вивчаю ісландську мову. Мері подобається Тому.
2,18,PERSON,том,24,27,False,У Тома брат та сестра. Том має брата і сестру.
3,21,PERSON,том,24,27,False,Том сів поруч із Мері. Том сів біля Мері.
4,23,PERSON,том,42,45,False,Том їсть рибу як мінімум раз на тиждень. Том ї...
5,28,PERSON,том,31,34,False,Скільки у Тома гітар? Скільки Том має гітар?
6,31,PERSON,том,19,22,False,Том був сміливий. Том був хоробрий.
7,41,PERSON,том,15,18,False,Том не бреше. Том має дві черепахи.
8,44,PERSON,том,19,22,False,Том був сміливий. Том був відважний.
9,48,PERSON,том,18,21,False,Том - мій кузен. Том - мій двоюрідний брат.


In [ ]:
# import json

# edge_cases = []

# for i, row in fp.iterrows():
#     edge_cases.append({
#         'id': f'fp_{i}',
#         'raw_text': row['text'],
#         'field_type': row['field_type'],
#         'expected_behavior': 'should NOT be detected as PERSON (false positive)'
#     })

# file_path = 'ie_edge_cases.jsonl'

# with open(file_path, 'a', encoding='utf-8') as f:
#     for case in edge_cases:
#         f.write(json.dumps(case, ensure_ascii=False) + '\n')